# 打印知识树：读取 `batch_qa.ipynb` 建好的索引库，以树状形式打印

`batch_qa.ipynb` 的「步骤 2：建库」用 `superindex.md_ingest.index_markdown` 把每个 Markdown 切成一棵**章节树**
（节点字段：`title` / `node_id` / `start_index` / `end_index` / 可选 `summary` / 子节点 `nodes`），
落盘在 `<WORK_DIR>/store/docs/<doc_id>/tree.json`（`WORK_DIR` 默认 `<ROOT_DIR>/data/notebook/<题集名>/`）。

本 Notebook **只读取已落盘的树来打印，不重新建库、不调用任何模型**。先运行 `batch_qa.ipynb` 到步骤 2 即可；
工作目录的算法与它一致（`NB_WORK_DIR` / `DATASET_PATH` 环境变量，可写进 `ROOT_DIR/.env`）。

本地运行：`uv sync --group notebook`，然后从上到下依次执行所有 cell。

## 环境

确定 `ROOT_DIR`（环境变量 > 向上找到的仓库根 > 当前目录）并加载 `.env`，与 `batch_qa.ipynb` 同一规则。

In [ ]:
# ───────────── 环境：确定 ROOT_DIR、加载 .env（与 batch_qa.ipynb 的"环境"cell 同一规则）─────────────
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv


def find_repo_root(start: Path) -> Path | None:
    """向上找含 pyproject.toml 和 superindex/ 包目录的目录（即本仓库根）；找不到返回 None。"""
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file() and (folder / "superindex").is_dir():
            return folder
    return None


_repo_root = find_repo_root(Path.cwd().resolve())
ROOT_DIR = Path(os.environ.get("ROOT_DIR", "").strip() or _repo_root or Path.cwd()).expanduser().resolve()
if _repo_root is not None:
    sys.path.insert(0, str(_repo_root))  # import superindex 用这份源码；不 chdir


def under_root(p: str | Path) -> Path:
    """绝对路径原样（resolve）；相对路径以 ROOT_DIR 为基准。"""
    p = Path(p).expanduser()
    return (p if p.is_absolute() else ROOT_DIR / p).resolve()


# .env：先 ROOT_DIR/.env，再当前目录的 .env；都 override=False，所以进程环境变量优先
for _env_file in dict.fromkeys([ROOT_DIR / ".env", Path.cwd().resolve() / ".env"]):
    if _env_file.is_file():
        load_dotenv(_env_file, override=False, encoding="utf-8")
        print(f".env     : {_env_file} 已加载")
    else:
        print(f".env     : {_env_file} 不存在")
print(f"ROOT_DIR : {ROOT_DIR}")

from superindex.engine.local_store import DocStore


## 配置

工作目录的算法与 `batch_qa.ipynb` 一致；其余是打印选项。

In [ ]:
# ───────────── 配置（所有可调项都在这一格）─────────────
# 每项都可用同名环境变量覆盖（方便命令行 nbconvert 执行；也可写进 ROOT_DIR/.env）。
def _env(name: str, default, cast=str):
    raw = os.environ.get(name)
    return default if raw is None or raw.strip() == "" else cast(raw.strip())


# 工作目录：与 batch_qa.ipynb 完全一致 —— NB_WORK_DIR，否则 ROOT_DIR/data/notebook/<题集名>
DATASET_PATH = under_root(_env("DATASET_PATH", "data/questions.jsonl"))  # 只用来取题集名（文件不必存在）
WORK_DIR = under_root(os.environ.get("NB_WORK_DIR", "").strip() or ROOT_DIR / "data" / "notebook" / DATASET_PATH.stem)
STORE = WORK_DIR / "store"  # 索引库（batch_qa.ipynb 步骤 2 的产物）

DOC: str | None = _env("TREE_DOC", None)         # 只打印名字或 doc_id 含该子串的文档（不分大小写）；None = 全部
MAX_DEPTH: int | None = _env("TREE_MAX_DEPTH", None, int)  # 最多打印几层（1 = 只打印顶层章节）；None = 不限
TEXT_WIDTH = _env("TREE_TEXT_WIDTH", 60, int)    # 节点标题 / 摘要各自最多显示多少字符，超出以 … 截断
SHOW_PAGES = True                                # 节点后面显示页码范围 (p.起-止)
SHOW_SUMMARY = True                              # 节点有 summary 时显示（建库时 INDEX_SUMMARY=True 才有）

print(f"工作目录 : {WORK_DIR}   [{'存在' if WORK_DIR.is_dir() else '不存在'}]")
print(f"索引库   : {STORE}   [{'存在' if STORE.is_dir() else '不存在'}]")

## 打印知识树

用 `DocStore` 读每个文档的 `tree.json`，按 `├──` / `└──` 连线逐层打印。

In [ ]:
# ───────────── 读取并打印 ─────────────
def clip(text: object, width: int) -> str:
    """压成单行并按 width 截断。"""
    s = " ".join(str(text or "").split())
    return s if len(s) <= width else s[: max(width - 1, 0)] + "…"


def count_nodes(nodes: list[dict]) -> int:
    return sum(1 + count_nodes(n.get("nodes") or []) for n in nodes)


def node_label(node: dict) -> str:
    """一个节点的显示文字：[node_id] 标题 (p.起-止) — 摘要。"""
    label = f"[{node.get('node_id', '?')}] {clip(node.get('title'), TEXT_WIDTH)}"
    if SHOW_PAGES and node.get("start_index") is not None:
        start, end = node["start_index"], node.get("end_index", node["start_index"])
        label += f" (p.{start})" if start == end else f" (p.{start}-{end})"
    if SHOW_SUMMARY and node.get("summary"):
        label += f" — {clip(node['summary'], TEXT_WIDTH)}"
    return label


def tree_lines(nodes: list[dict], prefix: str = "", depth: int = 1) -> list[str]:
    """tree 命令风格的连线；超过 MAX_DEPTH 的子树折叠成一行 '… (N 个节点)'。"""
    lines = []
    for i, node in enumerate(nodes):
        last = i == len(nodes) - 1
        lines.append(f"{prefix}{'└── ' if last else '├── '}{node_label(node)}")
        children = node.get("nodes") or []
        if not children:
            continue
        child_prefix = prefix + ("    " if last else "│   ")
        if MAX_DEPTH is not None and depth >= MAX_DEPTH:
            lines.append(f"{child_prefix}└── … （{count_nodes(children)} 个子节点未展开）")
        else:
            lines += tree_lines(children, child_prefix, depth + 1)
    return lines


store = DocStore(str(STORE))
docs = sorted((m for m in store.list_metas() if m.get("status") == "completed"), key=lambda m: m.get("name", ""))
if DOC:
    docs = [m for m in docs if DOC.lower() in m.get("name", "").lower() or DOC.lower() in m.get("id", "").lower()]
if not docs:
    raise RuntimeError(f"{STORE} 里没有可打印的文档（过滤条件 DOC={DOC!r}）。请先运行 batch_qa.ipynb 到步骤 2 建库，"
                       "或检查 NB_WORK_DIR / DATASET_PATH 是否与 batch_qa.ipynb 一致。")

print(f"共 {len(docs)} 个文档\n")
for meta in docs:
    tree = store.get_tree(meta["id"]) or []
    print(f"{meta.get('name')}  （doc_id {meta['id']}，{meta.get('pageNum')} 页，{count_nodes(tree)} 个节点）")
    print("\n".join(tree_lines(tree)) or "  （空树）")
    print()